## 5.2 Pytorch 模拟线性回归 - autograd手动实现

#### 1. 目标与思路
1. 我们要用最朴素的方式完成线性回归训练：`y = w * x + b`, `Loss = (ŷ - y)^2 / N`
2. 并且完全依赖 Autograd 来完成:
    * 自动求导（loss.backward()）
    * 手动更新参数（w -= lr * w.grad，b -= lr * b.grad）
    * 清空梯度（zero_()）

#### 2. 训练闭环流程（必须非常清楚）

每一轮 epoch 都做同样 5 步：
1. Forward：用当前 w,b 预测 y_hat
2. Loss：计算损失（MSE）
3. Backward：loss.backward() 自动算 w.grad 和 b.grad
4. Update：在 torch.no_grad() 下更新 w,b
5. Zero Grad：清空梯度，避免累加

#### 3. 完整流程

##### 3.1 数据准备，延用上一小节的数据

In [24]:
import torch
# 1. 定义x
x = torch.arange(1,51, dtype=torch.float32)

# 2. 定义w和b
true_w = 3
true_b = 2

# 3. 定义噪声
noise = torch.randn(x.shape) * 0.5

# 4. 定义y
y = true_w * x + true_b + noise

# 5. 打印x和y的形状
print(x.shape)
print(y.shape)

# 6. 将x和y转换为列向量
# 注意：在Pytorch中，输入和输出通常需要是二维的，即 (样本数, 特征数)，所以我们需要将x和y从一维转换为二维。
x = x.reshape(-1,1)
y = y.reshape(-1,1)

torch.Size([50])
torch.Size([50])


##### 3.2 初始化 w 和 b 参数，需要加入计算图
* w.shape = (1,1)
* b.shape = (1,)
* 这是为了让计算更自然：
    * x 是 (N,1)
    * w 是 (1,1) → 能广播到 (N,1)
    * b 是 (1) → 也能广播到 (N,1)
* 也可以把 b 写成 (1,1)，效果一样

In [25]:
w = torch.randn(1,1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.001

##### 3.3 训练

注意：x @ w + b 是最标准的线性回归写法 y = Xw + b，推荐使用。
* 不能写为 w @ x + b，因为`w.shape=(1,1)` 而 `x.shape=(50,1)`，维度不匹配（1与50）
* 也不能写为 w @ x.T + b，虽然可以合法相乘，但是此时结果`shape=(1,50)`, 而`y.shape=(50,1)`，结果不匹配

In [26]:
epochs = 30
for epoch in range(1, epochs+1):
    # 1. forward pass
    y_pred = x @ w + b # 广播：x(N,1)*w(1,1) -> (N,1)
    # 2. compute loss
    loss = ((y_pred - y) ** 2).mean()
    # 3. backward pass
    loss.backward()
    # 4. update weights
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    # 5. zero gradients
    w.grad.zero_()
    b.grad.zero_()
    # 6. 每10个epoch打印一次损失
    if epoch % 5 == 0:
        print("Epoch [{}/{}], w: {:.4f}, b: {:.4f}, loss: {:.4f}".format(epoch, epochs, w.item(), b.item(), loss.item()))

Epoch [5/30], w: 3.6757, b: 0.1196, loss: 648.2503
Epoch [10/30], w: 2.9329, b: 0.1019, loss: 24.7739
Epoch [15/30], w: 3.0750, b: 0.1105, loss: 1.9047
Epoch [20/30], w: 3.0476, b: 0.1141, loss: 1.0623
Epoch [25/30], w: 3.0527, b: 0.1186, loss: 1.0278
Epoch [30/30], w: 3.0516, b: 0.1229, loss: 1.0229
